# Notebook 1: Preprocessing & Baseline Models
PaySim Dataset — Track B Security & Fraud

Run this first. Saves preprocessed data for notebooks 2 and 3.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score,
    precision_score, recall_score
)
from imblearn.over_sampling import SMOTE
import warnings, joblib, os
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
print('Libraries loaded')

## 1. Load Data

In [ ]:
df = pd.read_csv('paysim_dataset.csv')
print(f'Full dataset: {len(df):,} rows')
print(f'Fraud rate: {df["isFraud"].mean()*100:.4f}%')
print(f'\nColumns: {df.columns.tolist()}')

## 2. EDA Plots (save for paper)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class distribution
labels = ['Legitimate', 'Fraud']
counts = df['isFraud'].value_counts().sort_index()
axes[0].bar(labels, counts, color=['#2196F3', '#F44336'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Transaction Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 50000, f'{v:,}', ha='center', fontweight='bold')

# Fraud by type
fraud_by_type = df.groupby('type')['isFraud'].agg(['sum', 'count'])
fraud_by_type['rate'] = fraud_by_type['sum'] / fraud_by_type['count'] * 100
axes[1].bar(fraud_by_type.index, fraud_by_type['rate'], color='#FF9800')
axes[1].set_title('Fraud Rate by Transaction Type (%)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

# Amount distribution fraud vs normal
axes[2].hist(df[df['isFraud']==0]['amount'].clip(upper=2e6), bins=50,
             alpha=0.6, label='Legitimate', color='#2196F3', density=True)
axes[2].hist(df[df['isFraud']==1]['amount'].clip(upper=2e6), bins=50,
             alpha=0.6, label='Fraud', color='#F44336', density=True)
axes[2].set_title('Transaction Amount Distribution')
axes[2].set_xlabel('Amount (clipped at 2M)')
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/eda_plots.png')

In [ ]:
# isFlaggedFraud baseline — show how bad rule-based is
print('=== isFlaggedFraud (existing rule-based system) ===')
print(f'Fraud cases in dataset:     {df["isFraud"].sum():,}')
print(f'Flagged by existing system: {df["isFlaggedFraud"].sum():,}')
flagged_caught = ((df['isFlaggedFraud']==1) & (df['isFraud']==1)).sum()
print(f'Correctly flagged (TP):     {flagged_caught:,}')
print(f'Rule-based Recall:          {flagged_caught/df["isFraud"].sum()*100:.2f}%')
print(f'\nThis means {100 - flagged_caught/df["isFraud"].sum()*100:.2f}% of fraud goes UNDETECTED by current rules.')

## 3. Feature Engineering

In [ ]:
# Filter to fraud-relevant transaction types only
df_model = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
print(f'After filtering to TRANSFER+CASH_OUT: {len(df_model):,} rows')
print(f'Fraud rate in filtered set: {df_model["isFraud"].mean()*100:.4f}%')

# Feature engineering
df_model['balance_drain']       = df_model['oldbalanceOrg'] - df_model['newbalanceOrig']
df_model['balance_drain_ratio'] = df_model['balance_drain'] / (df_model['oldbalanceOrg'] + 1)
df_model['amount_to_balance']   = df_model['amount'] / (df_model['oldbalanceOrg'] + 1)
df_model['dest_balance_delta']  = df_model['newbalanceDest'] - df_model['oldbalanceDest']
df_model['orig_drained']        = (df_model['newbalanceOrig'] == 0).astype(int)
df_model['is_transfer']         = (df_model['type'] == 'TRANSFER').astype(int)
df_model['step_hour']           = df_model['step'] % 24
df_model['step_hour_sin']       = np.sin(2 * np.pi * df_model['step_hour'] / 24)
df_model['step_hour_cos']       = np.cos(2 * np.pi * df_model['step_hour'] / 24)
df_model['log_amount']          = np.log1p(df_model['amount'])
df_model['log_old_balance']     = np.log1p(df_model['oldbalanceOrg'])

FEATURES = [
    'amount', 'log_amount',
    'oldbalanceOrg', 'log_old_balance', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'balance_drain', 'balance_drain_ratio',
    'amount_to_balance', 'dest_balance_delta',
    'orig_drained', 'is_transfer',
    'step_hour_sin', 'step_hour_cos'
]

print(f'\nFeatures ({len(FEATURES)}): {FEATURES}')

## 4. Train/Val/Test Split (Chronological)

In [ ]:
# Sort by step (time) — critical: no future leakage
df_model = df_model.sort_values('step').reset_index(drop=True)

n = len(df_model)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df_model.iloc[:train_end]
val_df   = df_model.iloc[train_end:val_end]
test_df  = df_model.iloc[val_end:]

print(f'Train: {len(train_df):,} rows | Fraud: {train_df["isFraud"].sum():,}')
print(f'Val:   {len(val_df):,} rows  | Fraud: {val_df["isFraud"].sum():,}')
print(f'Test:  {len(test_df):,} rows  | Fraud: {test_df["isFraud"].sum():,}')

X_train = train_df[FEATURES].values
y_train = train_df['isFraud'].values
X_val   = val_df[FEATURES].values
y_val   = val_df['isFraud'].values
X_test  = test_df[FEATURES].values
y_test  = test_df['isFraud'].values

## 5. Scaling + SMOTE (training only)

In [ ]:
# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# SMOTE on training only
print('Applying SMOTE to training set...')
sm = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)
print(f'After SMOTE — Train: {len(X_train_sm):,} | Fraud: {y_train_sm.sum():,} ({y_train_sm.mean()*100:.1f}%)')

# Save for other notebooks
joblib.dump(scaler, 'outputs/scaler.pkl')
np.save('outputs/X_train_sm.npy', X_train_sm)
np.save('outputs/y_train_sm.npy', y_train_sm)
np.save('outputs/X_val_sc.npy',   X_val_sc)
np.save('outputs/y_val.npy',      y_val)
np.save('outputs/X_test_sc.npy',  X_test_sc)
np.save('outputs/y_test.npy',     y_test)

# Also save raw test df for GNN notebook
test_df.to_csv('outputs/test_df.csv', index=False)
train_df.to_csv('outputs/train_df.csv', index=False)
print('Saved preprocessed data to outputs/')

## 6. Evaluation Helper

In [ ]:
def evaluate_model(name, y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    results = {
        'Model':     name,
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred), 4),
        'F1':        round(f1_score(y_true, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_pred_proba), 4),
        'PR-AUC':    round(average_precision_score(y_true, y_pred_proba), 4),
    }
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in results.items():
        if k != 'Model':
            print(f"  {k:<12}: {v}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Legitimate','Fraud'])}")
    return results

all_results = []

## 7. Baseline: isFlaggedFraud (Rule-Based System)

In [ ]:
# isFlaggedFraud as probability (it's binary 0/1)
y_flagged_test = test_df['isFlaggedFraud'].values.astype(float)
# Add tiny noise so PR-AUC doesn't crash on binary proba
y_flagged_proba = y_flagged_test + np.random.uniform(0, 1e-6, len(y_flagged_test))
r = evaluate_model('Rule-Based (isFlaggedFraud)', y_test, y_flagged_proba)
all_results.append(r)

## 8. Baseline: Logistic Regression

In [ ]:
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_sm, y_train_sm)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]
r = evaluate_model('Logistic Regression', y_test, lr_proba)
all_results.append(r)

## 9. Baseline: Random Forest

In [ ]:
print('Training Random Forest (this takes a few minutes)...')
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_sm, y_train_sm)
rf_proba = rf.predict_proba(X_test_sc)[:, 1]
r = evaluate_model('Random Forest', y_test, rf_proba)
all_results.append(r)

joblib.dump(rf, 'outputs/random_forest.pkl')

## 10. Feature Importance Plot

In [ ]:
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
plt.figure(figsize=(8, 6))
fi.plot(kind='barh', color='#2196F3')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150)
plt.show()
print('Saved: outputs/feature_importance.png')

## 11. Summary Table

In [ ]:
results_df = pd.DataFrame(all_results)
print('\n=== BASELINE RESULTS SUMMARY ===')
print(results_df.to_string(index=False))
results_df.to_csv('outputs/baseline_results.csv', index=False)
print('\nSaved: outputs/baseline_results.csv')
print('\nProceed to Notebook 2 (LSTM) and Notebook 3 (GNN)')